# Project 01 — demo mức 1: dự đoán quan hệ trong đồ thị cảnh

Chạy JupyterLab từ thư mục gốc repository. Notebook dùng VG150 PredCls: hộp và nhãn đối tượng có sẵn; mô hình dự đoán quan hệ. Chọn checkpoint GAT baseline để chạy ngay, hoặc đổi sang checkpoint GAT+WordNet sau khi train GPU. Phần biến đổi đồ thị theo câu chỉnh sửa thuộc mức 2 và chưa được minh họa như kết quả mức 1.


In [ ]:
import json
from pathlib import Path
from IPython.display import HTML, display
import torch
from scripts.evaluate_level1 import load_model
from scripts.extract_level1_examples import predict_graph
from scripts.train_mlp import make_dataset
from src.evaluation.demo_svg import scene_svg
from src.knowledge.transe.core import sha256

ROOT = Path.cwd()
assert (ROOT / 'README.md').exists(), 'Launch JupyterLab from the repository root'
DATA = ROOT / 'data/raw/vg150'
CHECKPOINT = ROOT / 'checkpoints/experiments/gat_bs512_seed42/run_0001/best.pt'
IMAGE_INDEX = 0  # First validation image; change only to select another demo image.
checkpoint = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
config = checkpoint['config']
if config.get('data_sha256'):
    actual = {name: sha256(DATA / name) for name in config['data_sha256']}
    assert actual == config['data_sha256'], 'Dataset differs from checkpoint inputs'
dataset = make_dataset(DATA, 'val', config['box_policy'])
graph = dataset[IMAGE_INDEX]
metadata = dataset.loader.image_metadata[dataset.loader.indices[IMAGE_INDEX]]
vocabulary = json.loads((DATA / 'VG-SGG-dicts-with-attri.json').read_text())
object_names = {value: name for name, value in vocabulary['label_to_idx'].items()}
predicate_names = {value: name for name, value in vocabulary['predicate_to_idx'].items()}
print('image_id:', graph.image_id, '| model:', config['model'], '| checkpoint epoch:', checkpoint['epoch'])
print('objects:', graph.num_nodes, '| annotated relations:', graph.gt_predicate.numel())
print('source image:', metadata.get('url'))


In [ ]:
model = load_model(checkpoint).eval()
with torch.inference_mode():
    logits = predict_graph(model, config['model'], graph, torch.device('cpu'))
scores, classes = logits.softmax(dim=-1)[:, 1:].max(dim=-1)
classes += 1
ranking = torch.argsort(scores, descending=True, stable=True).tolist()
candidate_pairs = graph.edge_label_index
predicted = [(int(candidate_pairs[0, i]), int(classes[i]), int(candidate_pairs[1, i]), float(scores[i])) for i in ranking[:50]]
annotated = sorted({(int(graph.gt_relation_index[0, i]), int(graph.gt_predicate[i]), int(graph.gt_relation_index[1, i]), None) for i in range(graph.gt_predicate.numel())})
image_path = DATA / 'images' / f'{graph.image_id}.jpg'
print('Optional local JPEG:', image_path, 'exists=', image_path.exists())
print('Ground-truth scene graph')
display(HTML(scene_svg(graph, annotated, object_names, predicate_names, width=metadata['width'], height=metadata['height'], image_path=image_path)))
print('Predicted scene graph: top 50 pairs, top-1 foreground predicate each')
display(HTML(scene_svg(graph, predicted, object_names, predicate_names, width=metadata['width'], height=metadata['height'], image_path=image_path)))
matched = {(s, p, o) for s, p, o, _ in predicted} & {(s, p, o) for s, p, o, _ in annotated}
print(f'Annotated relations recovered at top 50: {len(matched)}/{len(annotated)}')


**Ghi chú:** Nếu chưa có JPEG, vùng bên trái vẫn hiển thị hộp đối tượng trên nền trống. Sau khi chọn `image_id`, có thể tải đúng một ảnh bằng `python -m scripts.download_vg150_image --image-id ID` rồi chạy lại ô trên. Hình bên phải giới hạn 15 đối tượng và 12 quan hệ để dễ đọc; phép tính R@50 dùng toàn bộ candidate pairs. Dự đoán quan hệ có thể thiếu GT do xếp hạng hoặc chọn sai predicate.
